# MTPC — Full training on Google Colab (Phase 0 → 1 → 2)

Re-trains the byT5-small MTPC models **from scratch** on a GPU runtime, then runs a quick sanity check.

### Why this notebook trains WITHOUT `cheat`
`cheat` mode feeds the *entire* assistant answer into byT5's **bidirectional** encoder during
training. Because cross-attention is not causally masked, the decoder can read the token it is
supposed to predict straight from the encoder — so the model learns to **copy**, not predict.
That gives a deceptively low training loss but produces **byte-salad at inference**, where the
answer is not yet in the encoder. Standard SFT (cheat off) masks the answer in the encoder, so
the model actually learns to predict. **All cells below run without `--cheat`.**

Run order: **1 → 2 → 3 → 4 → (5) → 6**. Set *Runtime ▸ Change runtime type ▸ GPU* first.

## 1. Mount Drive & clone the repo

In [ ]:
import os, subprocess
# --- GPU check ---
print(subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True).stdout
      or "!! No GPU detected — set Runtime > Change runtime type > GPU and re-run.")

# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- Clone (or update) the project on the Drive so checkpoints persist ---
PROJECT_DIR = '/content/drive/MyDrive/MTPC'        # change if you want a different location
REPO_URL    = 'https://github.com/lorenzoAllegrini/MTPC.git'
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} "{PROJECT_DIR}"
else:
    !cd "{PROJECT_DIR}" && git pull
%cd {PROJECT_DIR}

## 2. Install dependencies
Colab already ships a compatible `torch`; we only add the HF stack.

In [ ]:
!pip install -q -U transformers peft datasets accelerate

## 3. Configuration
Tune for your GPU. The CP/HMM heads are memory-heavy (`ranks*window*vocab` outputs), so on a
free **T4** keep `BATCH_SIZE` small and `MAX_LEN` modest; on **A100** you can raise both.
Gradient checkpointing is enabled automatically inside `training.py`.

In [ ]:
MODEL_ID    = "google/byt5-small"
WINDOW_SIZE = 6
RANKS       = 32
MAX_SAMPLES = 20000     # flan_v2 examples after filtering (more = better model, longer training)
MAX_LEN     = 1024      # bytes per example (each example is padded to this length)
BATCH_SIZE  = 4         # T4: 2-4 | A100: 8-16  (lower this first if you hit CUDA OOM)
# cheat stays OFF on purpose (no --cheat flag below) -> no encoder answer-leakage.

## 4. Phase 0 + 1 + 2 — backbone SFT, FF warm-up, FF joint
This single run trains the shared backbone (Phase 0), warms up the Feed-Forward head
(Phase 1), and joint-trains the FF draft (Phase 2). It also produces the **verifier** backbone
(`saved_models/byt5_standard_lora_phase0`). First run will download & tokenize Tülu-3 (slow,
cached to Drive afterwards).

In [ ]:
!cd src && python training.py \
  --model_id {MODEL_ID} --head ff \
  --window_size {WINDOW_SIZE} --ranks {RANKS} \
  --max_samples {MAX_SAMPLES} --max_len {MAX_LEN} --batch_size {BATCH_SIZE} \
  --skip_phase_0 false --skip_phase_1 false --skip_phase_2 false

## 5. Phase 2 — CP head
Reuses the Phase-0 backbone and transfers the Phase-1 FF emissions into the CP circuit, then
joint-trains it. (Skips Phase 0/1, which were just done above.)

In [ ]:
!cd src && python training.py \
  --model_id {MODEL_ID} --head cp \
  --window_size {WINDOW_SIZE} --ranks {RANKS} \
  --max_samples {MAX_SAMPLES} --max_len {MAX_LEN} --batch_size {BATCH_SIZE} \
  --skip_phase_0 true --skip_phase_1 true --skip_phase_2 false

## 6. (optional) Phase 2 — HMM head

In [ ]:
!cd src && python training.py \
  --model_id {MODEL_ID} --head hmm \
  --window_size {WINDOW_SIZE} --ranks {RANKS} \
  --max_samples {MAX_SAMPLES} --max_len {MAX_LEN} --batch_size {BATCH_SIZE} \
  --skip_phase_0 true --skip_phase_1 true --skip_phase_2 false

## 7. Sanity check — honest (non-cheat) prediction
With cheat off, the qualitative test loads with the standard masked encoder. The argmax should
now be a real *prediction* (not a verbatim copy of the ground truth) and the per-step loss
should be well below the random baseline. If you instead see argmax == ground truth exactly,
cheat leaked somewhere.

In [ ]:
!cd src && python inference.py --head cp --window_size {WINDOW_SIZE} --num_samples 5

## 8. Done
Trained adapters/heads are saved on your Drive under `MTPC/saved_models/`:
- `byt5_standard_lora_phase0/`  → verifier backbone
- `mtp_head_{ff,cp,hmm}_w6_final.pth` + `lora_{ff,cp,hmm}_w6/`  → draft heads + adapters

For the R speculative-decoding benchmark, pull the repo locally and run
`speculative_inference_testing.R` with **`CHEAT = FALSE`** (already the default now).